# 第 44 章：Capstone 课程日历与里程碑

这个 notebook 用一个极小的 course-calendar table，对比“只按 workload 和周数判断”的 naive baseline 和“先看风险、最终交付、学生交付物与教师检查项”的课程周历 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_course_calendar_demo.csv')


def load_calendar(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['week'] = int(row['week'])
            row['workload_score'] = float(row['workload_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_calendar(DATA_PATH)
print(f'Loaded {len(rows)} course-calendar weeks from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Phase counts:', ordered_counts(rows, 'phase'))
print('Risk status:', ordered_counts(rows, 'risk_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
def naive_route_week(row):
    if row['workload_score'] > 0.70:
        return 'instructor_intervention'
    if row['week'] >= 15:
        return 'final_delivery_week'
    return 'regular_instruction'


naive_rows = []
for row in rows:
    routed = dict(row)
    routed['naive_route'] = naive_route_week(row)
    naive_rows.append(routed)

naive_accuracy = sum(
    row['naive_route'] == row['reference_route']
    for row in naive_rows
) / len(naive_rows)

print('Naive workload/week routing:')
for row in naive_rows:
    print(
        f"  week {row['week']:02d}: naive={row['naive_route']}, "
        f"reference={row['reference_route']}, milestone={row['milestone']}"
    )
print('Naive route accuracy:', round(naive_accuracy, 3))


In [ ]:
FINAL_PHASES = {'final_delivery', 'showcase'}


def route_calendar_week(row):
    if row['risk_status'] == 'high':
        return 'instructor_intervention', 'high-risk week needs active instructor intervention'
    if row['phase'] in FINAL_PHASES:
        return 'final_delivery_week', 'final report, presentation, archive, or signoff week'
    if row['student_deliverable'] != 'none':
        return 'checkpoint_week', 'student deliverable creates a visible project state gate'
    if row['instructor_check'] != 'none':
        return 'checkpoint_week', 'instructor check should be explicit on the calendar'
    return 'regular_instruction', 'ordinary instruction and project work week'


routed_rows = []
for row in rows:
    route, reason = route_calendar_week(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Structured calendar routes:')
for row in routed_rows:
    print(
        f"  week {row['week']:02d}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
regular_weeks = [row for row in routed_rows if row['route'] == 'regular_instruction']
checkpoint_weeks = [row for row in routed_rows if row['route'] == 'checkpoint_week']
intervention_weeks = [row for row in routed_rows if row['route'] == 'instructor_intervention']
final_weeks = [row for row in routed_rows if row['route'] == 'final_delivery_week']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)

print('Regular instruction weeks:', [row['week'] for row in regular_weeks])
print('Checkpoint weeks:', [row['week'] for row in checkpoint_weeks])
print('Instructor-intervention weeks:', [row['week'] for row in intervention_weeks])
print('Final-delivery weeks:', [row['week'] for row in final_weeks])
print('Workflow route accuracy:', round(workflow_accuracy, 3))


In [ ]:
def calendar_team_checklist(row):
    steps = []
    if row['risk_status'] == 'high':
        steps.append('安排教师或课程团队主动介入，不要只等学生下次提交。')
    if row['student_deliverable'] != 'none':
        steps.append(f"检查学生交付物：{row['student_deliverable']}。")
    if row['instructor_check'] != 'none':
        steps.append(f"完成教师检查项：{row['instructor_check']}。")
    if row['ai_policy_focus']:
        steps.append(f"提醒 AI 使用规范焦点：{row['ai_policy_focus']}。")
    if row['phase'] in FINAL_PHASES:
        steps.append('确认报告、notebook、AI-use statement、展示、归档和 signoff 都已进入最终检查。')
    return steps


for row in routed_rows:
    if row['route'] in {'instructor_intervention', 'final_delivery_week'}:
        print(f"\nWeek {row['week']} -> {row['route']} ({row['milestone']})")
        for step in calendar_team_checklist(row):
            print(' -', step)


In [ ]:
must_keep_milestones = [
    'proposal card and data boundary',
    'one-week pilot and baseline',
    'midpoint project board review',
    'validation and error-analysis checkpoint',
    'notebook freeze and reproducibility snapshot',
    'final package, presentation, archive, and signoff',
]

calendar_assistant_prompt = '''你是我的 capstone 课程日历助手。

任务：
1. 先阅读 course-calendar table，不要只按 workload 排序；
2. 先识别 high-risk 周和 final-delivery 周；
3. 再检查每周是否有 student deliverable 或 instructor check；
4. 把每周 route 到 regular_instruction、checkpoint_week、
   instructor_intervention 或 final_delivery_week；
5. 如果要压缩到 12 周，说明哪些 checkpoint 可以合并，哪些不能删除。

输出格式：
- 16-week route table
- High-risk intervention weeks
- Final-delivery weeks
- 12-week compression notes
- Instructor checklist
'''

print('Must-keep milestones for a 12-week compression:')
for item in must_keep_milestones:
    print(' -', item)

print('\nAssistant prompt:')
print(calendar_assistant_prompt)


In [ ]:
calendar_snapshot = {
    'dataset': DATA_PATH.name,
    'n_weeks': len(rows),
    'naive_route_accuracy': round(naive_accuracy, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'checkpoint_weeks': [row['week'] for row in checkpoint_weeks],
    'intervention_weeks': [row['week'] for row in intervention_weeks],
    'final_delivery_weeks': [row['week'] for row in final_weeks],
    'python_version': platform.python_version(),
}

print('Calendar snapshot:')
for key, value in calendar_snapshot.items():
    print(f'  {key}: {value}')
